# 01 Dataset audit

Explore MedQuAD candidates and review queues one cell at a time. This notebook only reads inputs; it does not approve medical content or modify source files.

In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
print('Project root:', ROOT)

Project root: /home/hamse/alu/ml/diabetes-education-qa


In [2]:
queue=pd.read_csv(ROOT/'data/review/medquad_review_queue.csv')
scope=pd.read_csv(ROOT/'data/review/medquad_scope_screen.csv')
print('Review queue:',queue.shape,'| scope screen:',scope.shape)
queue.head()

Review queue: (175, 19) | scope screen: (175, 21)


,doc_id,qa_id,source,source_collection,source_url,focus,question_type,question,answer,scope_review,review_status,source_urls,source_collections,scope_decision,source_support_review,currentness_review,safety_review,approved_for_training,reviewer_notes
0,52,0000052-1,NHLBI,8_NHLBI_QA_XML,http://www.nhlbi.nih.gov/health/health-topics/...,Diabetic Heart Disease,information,What is (are) Diabetic Heart Disease ?,"The term ""diabetic heart disease"" (DHD) refers...",candidate_general_adult_education,unreviewed,http://www.nhlbi.nih.gov/health/health-topics/...,8_NHLBI_QA_XML,NaN,NaN,NaN,NaN,NaN,NaN
1,52,0000052-2,NHLBI,8_NHLBI_QA_XML,http://www.nhlbi.nih.gov/health/health-topics/...,Diabetic Heart Disease,causes,What causes Diabetic Heart Disease ?,"At least four complex processes, alone or comb...",candidate_general_adult_education,unreviewed,http://www.nhlbi.nih.gov/health/health-topics/...,8_NHLBI_QA_XML,NaN,NaN,NaN,NaN,NaN,NaN
2,52,0000052-3,NHLBI,8_NHLBI_QA_XML,http://www.nhlbi.nih.gov/health/health-topics/...,Diabetic Heart Disease,susceptibility,Who is at risk for Diabetic Heart Disease? ?,People who have type 1 or type 2 diabetes are ...,candidate_general_adult_education,unreviewed,http://www.nhlbi.nih.gov/health/health-topics/...,8_NHLBI_QA_XML,NaN,NaN,NaN,NaN,NaN,NaN
3,52,0000052-4,NHLBI,8_NHLBI_QA_XML,http://www.nhlbi.nih.gov/health/health-topics/...,Diabetic Heart Disease,symptoms,What are the symptoms of Diabetic Heart Disease ?,Some people who have diabetic heart disease (D...,candidate_general_adult_education,unreviewed,http://www.nhlbi.nih.gov/health/health-topics/...,8_NHLBI_QA_XML,NaN,NaN,NaN,NaN,NaN,NaN
4,52,0000052-5,NHLBI,8_NHLBI_QA_XML,http://www.nhlbi.nih.gov/health/health-topics/...,Diabetic Heart Disease,exams and tests,How to diagnose Diabetic Heart Disease ?,Your doctor will diagnose diabetic heart disea...,candidate_general_adult_education,unreviewed,http://www.nhlbi.nih.gov/health/health-topics/...,8_NHLBI_QA_XML,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
scope.scope_decision_draft.value_counts(dropna=False).to_frame('rows')

,rows
scope_decision_draft,
in_scope_pending_content_review,156
exclude_pending_confirmation,19


In [4]:
scope_only=queue.merge(scope[['qa_id','scope_decision_draft']],on='qa_id',how='inner')
scope_only=scope_only[scope_only.scope_decision_draft.eq('in_scope_pending_content_review')]
print('Topic-screened rows:',len(scope_only),'| source documents:',scope_only.doc_id.nunique(),'| unique questions:',scope_only.question.str.casefold().str.strip().nunique())
scope_only[['qa_id','question','focus','source_url']].head(10)

Topic-screened rows: 156 | source documents: 35 | unique questions: 99


,qa_id,question,focus,source_url
0,0000052-1,What is (are) Diabetic Heart Disease ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
1,0000052-2,What causes Diabetic Heart Disease ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
2,0000052-3,Who is at risk for Diabetic Heart Disease? ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
3,0000052-4,What are the symptoms of Diabetic Heart Disease ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
4,0000052-5,How to diagnose Diabetic Heart Disease ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
5,0000052-6,What are the treatments for Diabetic Heart Dis...,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
6,0000052-7,How to prevent Diabetic Heart Disease ?,Diabetic Heart Disease,http://www.nhlbi.nih.gov/health/health-topics/...
7,0000106-1,What is (are) Diabetic Neuropathy ?,Diabetic Neuropathy,http://www.ninds.nih.gov/disorders/diabetic/di...
8,0000106-2,What are the treatments for Diabetic Neuropathy ?,Diabetic Neuropathy,http://www.ninds.nih.gov/disorders/diabetic/di...
9,0000106-3,What is the outlook for Diabetic Neuropathy ?,Diabetic Neuropathy,http://www.ninds.nih.gov/disorders/diabetic/di...


## Review status

Topic screening is not source verification, safety review, or clinical approval. Human review fields remain pending.